In [1]:
from dotenv import load_dotenv
from databricks.sdk import WorkspaceClient

load_dotenv()
w = WorkspaceClient()
C, S = "workspace", "ai_data_team"
wh = next(iter(w.warehouses.list()))

def run(sql):
    r = w.statement_execution.execute_statement(
        statement=sql, warehouse_id=wh.id, wait_timeout="50s")
    if r.status.state.value != "SUCCEEDED":
        raise RuntimeError(r.status.error)
    return r

def one(sql):
    return run(sql).result.data_array[0][0]

# 1. Build clean tables
run(f"""CREATE OR REPLACE TABLE {C}.{S}.clean_customers AS
        SELECT * EXCEPT (_rescued_data, registration_date),
               CAST(registration_date AS DATE) AS registration_date
        FROM {C}.{S}.customers
        WHERE customer_id IS NOT NULL
        QUALIFY ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY registration_date) = 1""")

run(f"""CREATE OR REPLACE TABLE {C}.{S}.clean_orders AS
        SELECT * EXCEPT (_rescued_data, order_date, delivery_date),
               CAST(order_date AS DATE) AS order_date,
               CAST(delivery_date AS DATE) AS delivery_date
        FROM {C}.{S}.orders
        WHERE order_id IS NOT NULL
        QUALIFY ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY order_date) = 1""")

run(f"""CREATE OR REPLACE TABLE {C}.{S}.clean_monthly_revenue AS
        SELECT * EXCEPT (_rescued_data) FROM {C}.{S}.monthly_revenue""")

run(f"""CREATE OR REPLACE TABLE {C}.{S}.clean_product_summary AS
        SELECT * EXCEPT (_rescued_data) FROM {C}.{S}.product_summary""")

# 2. Data quality report
t = lambda n: f"{C}.{S}.{n}"
checks = {
  "customers: raw -> clean rows": (one(f"SELECT COUNT(*) FROM {t('customers')}"), one(f"SELECT COUNT(*) FROM {t('clean_customers')}")),
  "orders: raw -> clean rows": (one(f"SELECT COUNT(*) FROM {t('orders')}"), one(f"SELECT COUNT(*) FROM {t('clean_orders')}")),
  "orders with unknown customer_id": one(f"SELECT COUNT(*) FROM {t('clean_orders')} o LEFT JOIN {t('clean_customers')} c USING (customer_id) WHERE c.customer_id IS NULL"),
  "orders dated before customer registration": one(f"SELECT COUNT(*) FROM {t('clean_orders')} o JOIN {t('clean_customers')} c USING (customer_id) WHERE o.order_date < c.registration_date"),
  "orders where total != subtotal - discount + shipping + tax": one(f"SELECT COUNT(*) FROM {t('clean_orders')} WHERE ABS(total_amount_usd - (subtotal_usd - discount_amount_usd + shipping_fee_usd + tax_amount_usd)) > 0.05"),
  "orders with null total": one(f"SELECT COUNT(*) FROM {t('clean_orders')} WHERE total_amount_usd IS NULL"),
  "order date range": (one(f"SELECT MIN(order_date) FROM {t('clean_orders')}"), one(f"SELECT MAX(order_date) FROM {t('clean_orders')}")),
}
for k, v in checks.items():
    print(f"{k}: {v}")

customers: raw -> clean rows: ('8000', '8000')
orders: raw -> clean rows: ('25000', '25000')
orders with unknown customer_id: 0
orders dated before customer registration: 18592
orders where total != subtotal - discount + shipping + tax: 0
orders with null total: 0
order date range: ('2020-01-01', '2026-03-30')
